#Evaluation of VIS and IR pedestrian detection baselines

Purpose
Evaluate the selected VIS and IR checkpoints using identical settings
on the same 1,995 development-validation pairs.

Checkpoints
VIS: epoch8.pt — ninth training epoch
IR: epoch9.pt — tenth training epoch

These checkpoints were selected using the highest validation AP50–95.
Both detectors completed 10 training epochs.

Scope
This is development-validation evaluation, not held-out test performance.
The official test data remains unused.

Outputs
Verified comparison table, evaluation settings, saved predictions,
and paired examples with ground-truth boxes.

Our equal-weight fusion baseline is a later experiment.

## Mount Drive and Define Paths

In [1]:
from pathlib import Path
from google.colab import drive

# connect this notebook's runtime to google drive
drive.mount("/content/drive")

SHARED_ROOT = Path("/content/drive/MyDrive/EO_IR_Capstone")
assert SHARED_ROOT.is_dir(), (
    "Shared folder not found. Check the Google account and Drive shortcut."
)

# This is the recovered archive we already built and verified
ARCHIVE_DRIVE = (
    SHARED_ROOT
    / "data/prepared/recovery_v1/llvip_milestone_recovered_v1.zip"
)
ARCHIVE_SHA256 = (
    "5c5274dabf94113d66bf86fc81cb86a2b71"
    "a0adfe2e19d5322c28d356b06e07e"
)

assert ARCHIVE_DRIVE.is_file(), f"Archive missing: {ARCHIVE_DRIVE}"

# Keep runtime paths stable so interrupted runs can resume in a fresh session
YOLO_DIR = Path("/content/yolov5")
EXTRACT_DIR = Path("/content/eo_ir_recovery/verified_bundle")
DATA_ROOT = EXTRACT_DIR / "eo_ir_data"

# keep this experiment ID unchanged when resuming these runs
# Use a different ID only when deliberately starting a new experiment
EXPERIMENT_ID = "baselines_recovery_v1_10ep_seed42_v1"
EXPERIMENT_DIR = SHARED_ROOT / "runs" / EXPERIMENT_ID
CONFIG_DIR = EXPERIMENT_DIR / "configs"

print("Shared archive", ARCHIVE_DRIVE)
print("Experiment outputs:", EXPERIMENT_DIR)

Mounted at /content/drive
Shared archive /content/drive/MyDrive/EO_IR_Capstone/data/prepared/recovery_v1/llvip_milestone_recovered_v1.zip
Experiment outputs: /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1


## Install and Check Detector

In [7]:
import importlib.metadata
import json
import subprocess
import sys
import torch

# Evaluation can use CPU when Colab GPU access is unavailable.
# Use full precision on CPU; enable half precision only on GPU.
EVAL_DEVICE = "0" if torch.cuda.is_available() else "cpu"
EVAL_HALF = torch.cuda.is_available()
hardware_name = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

print("Evaluation hardware:", hardware_name)
YOLO_COMMIT = "f9578796b8a64889aa9e66dc06914d19f40d338e"

#Clone public detector code if this is a fresh runtime
if not YOLO_DIR.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/ultralytics/yolov5.git",
            str(YOLO_DIR),
        ],
        check=True,
  )

# use the same detector revision as the successful pilot
subprocess.run(
    ["git", "-C", str(YOLO_DIR), "checkout", "--detach", YOLO_COMMIT],
    check=True,
)

# Match the Ultralytics version recorded during baseline training.
# Keep other dependencies unchanged because the audit confirmed they match.
required_ultralytics = "8.4.188"

try:
    installed_ultralytics = importlib.metadata.version("ultralytics")
except importlib.metadata.PackageNotFoundError:
    installed_ultralytics = None

if installed_ultralytics != required_ultralytics:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-deps",
            f"ultralytics=={required_ultralytics}",
        ],
        check=True,
    )

assert (
    importlib.metadata.version("ultralytics")
    == required_ultralytics
)


#Preserve installed versions of key packages while satisfying requirements
constraints = Path("/content/eo_ir_runtime_constraints.txt")
protected_packages = [
    "torch", "torchvision", "numpy", "pandas", "pillow", "ultralytics"
]
pins = []

for package in protected_packages:
    try:
        version = importlib.metadata.version(package)
        # Match the Ultralytics version actually used during training.
        # Preserve this runtime's CPU builds of torch and torchvision.
        if package == "ultralytics":
            version = "8.4.188"

        pins.append(f"{package}=={version}")
    except importlib.metadata.PackageNotFoundError:
        pass

constraints.write_text("\n".join(pins) + "\n")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(YOLO_DIR / "requirements.txt"),
        "-c", str(constraints),
    ],
    check=True,
)

# Verify the checkout and record the actual environment used for these runs
actual_commit = subprocess.check_output(
    ["git", "-C", str(YOLO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()
assert actual_commit == YOLO_COMMIT

ENVIRONMENT = {
    "python": sys.version.split()[0],
    "gpu": hardware_name,
    "yolov5_commit": actual_commit,
    "packages": {
        package: importlib.metadata.version(package)
        for package in protected_packages
    },
}

print(json.dumps(ENVIRONMENT, indent=2))

Evaluation hardware: Tesla T4
{
  "python": "3.13.16",
  "gpu": "Tesla T4",
  "yolov5_commit": "f9578796b8a64889aa9e66dc06914d19f40d338e",
  "packages": {
    "torch": "2.11.0+cu130",
    "torchvision": "0.26.0+cu130",
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "pillow": "11.3.0",
    "ultralytics": "8.4.188"
  }
}


## Load the Recovered Archive

In [3]:
import hashlib
import shutil
import zipfile

def sha256_file(path):
    """Calculate a file's checksum without loading it all into memory."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Prefer the archive left by our recovery work if this runtime has it.
recovery_copy = Path(
    "/content/eo_ir_recovery/llvip_milestone_recovered_v1.zip"
)
download_copy = Path("/content/llvip_milestone_recovered_v1.zip")

if recovery_copy.is_file():
    local_archive = recovery_copy
else:
    local_archive = download_copy
    if not local_archive.is_file():
        print("Copying recovered archive from Drive...", flush=True)
        shutil.copyfile(ARCHIVE_DRIVE, local_archive)

print("Verifying archive checksum...", flush=True)
assert sha256_file(local_archive) == ARCHIVE_SHA256, (
    "Archive checksum mismatch. Stop before training."
)

# Extract the verified bundle into local storage for fast image loading.
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print("Extracting prepared data...", flush=True)

with zipfile.ZipFile(local_archive) as archive:
    archive.extractall(EXTRACT_DIR)

assert DATA_ROOT.is_dir()
print("Prepared data available at:", DATA_ROOT)

Copying recovered archive from Drive...
Verifying archive checksum...
Extracting prepared data...
Prepared data available at: /content/eo_ir_recovery/verified_bundle/eo_ir_data


##Validate the Data

In [8]:
import json

inventory = json.loads((DATA_ROOT / "inventory.json").read_text())
assert inventory["split"]["train_pairs"] == 10028
assert inventory["split"]["val_pairs"] == 1995

pair_sets = {}

for modality in ["vis", "ir"]:
    for split, expected in [("train", 10028), ("val", 1995)]:
        entries = (DATA_ROOT / f"{modality}_{split}.txt").read_text().split()
        image_paths = [DATA_ROOT / entry for entry in entries]
        ids = [path.stem for path in image_paths]

        # Check membership, duplicates, and the presence of corresponding labels.
        assert len(ids) == expected
        assert len(set(ids)) == expected
        assert all(path.is_file() for path in image_paths)
        assert all(
            (DATA_ROOT / modality / "labels" / f"{p}.txt").is_file()
            for p in ids
        )

        pair_sets[(modality, split)] = set(ids)

assert pair_sets[("vis", "train")] == pair_sets[("ir", "train")]
assert pair_sets[("vis", "val")] == pair_sets[("ir", "val")]
assert not pair_sets[("vis", "train")] & pair_sets[("vis", "val")]
assert not {"100030", "100033"} & (
    pair_sets[("vis", "train")] | pair_sets[("vis", "val")]
)

# Import the loader from the pinned detector checkout.
sys.path.insert(0, str(YOLO_DIR))
from utils.dataloaders import LoadImagesAndLabels

for modality in ["vis", "ir"]:
    for split in ["train", "val"]:
        loaded = LoadImagesAndLabels(
            str(DATA_ROOT / f"{modality}_{split}.txt"),
            img_size=640,
            batch_size=8,
            augment=False,
        )
        assert len(loaded.im_files) == len(pair_sets[(modality, split)])
        print(f"{modality}_{split}: passed")

print("\nDATA VALIDATION PASSED")

Scanning /content/eo_ir_recovery/verified_bundle/eo_ir_data/vis_train.cache... 10028 images, 0 backgrounds, 0 corrupt
vis_train: passed
Scanning /content/eo_ir_recovery/verified_bundle/eo_ir_data/vis_val.cache... 1995 images, 0 backgrounds, 0 corrupt
vis_val: passed
Scanning /content/eo_ir_recovery/verified_bundle/eo_ir_data/ir_train.cache... 10028 images, 0 backgrounds, 0 corrupt
ir_train: passed
Scanning /content/eo_ir_recovery/verified_bundle/eo_ir_data/ir_val.cache... 1995 images, 0 backgrounds, 0 corrupt
ir_val: passed

DATA VALIDATION PASSED


##Verify Saved Experiment

In [9]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import yaml

# Load the original protocol without changing it.
protocol_file = EXPERIMENT_DIR / "protocol.json"
assert protocol_file.is_file(), protocol_file
protocol = json.loads(protocol_file.read_text())

# Evaluation must use the exact recovered dataset used for training.
assert protocol["dataset_sha256"] == ARCHIVE_SHA256, (
    "The dataset archive differs from the training protocol."
)

# CPU evaluation is allowed, but software differences need inspection.
training_environment = protocol["environment"]

print("Training hardware:", training_environment.get("gpu"))
print("Evaluation hardware:", ENVIRONMENT.get("gpu"))

software_differences = []

for field in ["python", "yolov5_commit", "packages"]:
    training_value = training_environment.get(field)
    evaluation_value = ENVIRONMENT.get(field)

    if training_value != evaluation_value:
        software_differences.append(field)
        print(f"\nSOFTWARE DIFFERENCE: {field}")
        print("Training:")
        print(json.dumps(training_value, indent=2))
        print("Evaluation:")
        print(json.dumps(evaluation_value, indent=2))

assert not software_differences, (
    "Software differences require inspection: "
    + ", ".join(software_differences)
)

print("Software environment checks passed.\n")

CHECKPOINTS = {}
training_selected_rows = {}
actual_options = {}
actual_hyperparameters = {}

metric_columns = [
    "metrics/precision",
    "metrics/recall",
    "metrics/mAP_0.5",
    "metrics/mAP_0.5:0.95",
]

for modality in ["vis", "ir"]:
    run_dir = EXPERIMENT_DIR / f"{modality}_baseline"

    # Inspect the actual training settings saved by YOLOv5.
    options = yaml.safe_load((run_dir / "opt.yaml").read_text())
    hyperparameters = yaml.safe_load((run_dir / "hyp.yaml").read_text())

    assert options["epochs"] == 10, f"{modality}: unexpected epoch setting"
    assert options["batch_size"] == 8, f"{modality}: unexpected batch size"
    assert options["seed"] == 42, f"{modality}: unexpected seed"
    assert options["imgsz"] in (640, [640], [640, 640]), (
        f"{modality}: unexpected image size"
    )

    actual_options[modality] = options
    actual_hyperparameters[modality] = hyperparameters

    # Confirm all 10 epochs were recorded and their metrics are finite.
    results = pd.read_csv(run_dir / "results.csv")
    results.columns = results.columns.str.strip()

    assert len(results) == 10, f"{modality}: incomplete training history"
    assert list(results["epoch"].astype(int)) == list(range(10)), (
        f"{modality}: unexpected epoch sequence"
    )
    assert np.isfinite(
        results[metric_columns].to_numpy(dtype=float)
    ).all(), f"{modality}: non-finite metric found"

    # Reproduce our selection rule independently from the saved CSV.
    # Highest AP50–95 wins; the earlier epoch breaks an exact tie.
    selected_row = results.sort_values(
        ["metrics/mAP_0.5:0.95", "epoch"],
        ascending=[False, True],
    ).iloc[0]

    selected_epoch = int(selected_row["epoch"])
    checkpoint = run_dir / "weights" / f"epoch{selected_epoch}.pt"
    assert checkpoint.is_file(), f"Checkpoint missing: {checkpoint}"

    # Verify that the existing selection record agrees with this audit.
    selection = json.loads((run_dir / "selection.json").read_text())
    assert selection["selected_epoch_zero_based"] == selected_epoch
    assert Path(selection["selected_checkpoint"]) == checkpoint

    CHECKPOINTS[modality] = checkpoint
    training_selected_rows[modality] = selected_row

    print(
        f"{modality.upper()}: 10 epochs completed; "
        f"selected epoch {selected_epoch + 1}/10"
    )
    print("Checkpoint:", checkpoint)

# Check that optimization and augmentation settings matched across modalities.
assert actual_hyperparameters["vis"] == actual_hyperparameters["ir"], (
    "VIS and IR used different hyperparameter settings."
)

for setting in [
    "epochs",
    "batch_size",
    "imgsz",
    "seed",
    "workers",
    "optimizer",
    "single_cls",
]:
    assert actual_options["vis"][setting] == actual_options["ir"][setting], (
        f"Training settings differ: {setting}"
    )

print("\nEXPERIMENT CHECKS PASSED")

Training hardware: Tesla T4
Evaluation hardware: Tesla T4
Software environment checks passed.

VIS: 10 epochs completed; selected epoch 9/10
Checkpoint: /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1/vis_baseline/weights/epoch8.pt
IR: 10 epochs completed; selected epoch 10/10
Checkpoint: /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1/ir_baseline/weights/epoch9.pt

EXPERIMENT CHECKS PASSED


##Define Evaluation Settings

In [10]:
from datetime import datetime, timezone

# A unique folder preserves previous evaluation attempts and predictions.
evaluation_id = "selected_" + datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S"
)
EVALUATION_DIR = EXPERIMENT_DIR / "evaluation" / evaluation_id
EVALUATION_DIR.mkdir(parents=True, exist_ok=False)

EVALUATION_SETTINGS = {
    "batch_size": 16,
    "imgsz": 640,
    "conf_thres": 0.001,
    "iou_thres": 0.60,
    "max_det": 300,
    "task": "val",
    "device": "0",
    "workers": 2,
    "single_cls": False,
    "augment": False,
    "half": True,
}

# This separate threshold will be used for our later example figures.
# It does not replace the low confidence cutoff used to calculate AP.
DISPLAY_CONFIDENCE = 0.25

evaluation_record = {
    "settings": EVALUATION_SETTINGS,
    "display_confidence_for_examples": DISPLAY_CONFIDENCE,
    "dataset_sha256": ARCHIVE_SHA256,
    "checkpoint_sha256": {
        modality: sha256_file(path)
        for modality, path in CHECKPOINTS.items()
    },
    "precision_recall_definition": (
        "Evaluator-selected operating point based on maximum smoothed "
        "validation F1; the selected confidence can differ by modality."
    ),
    "evaluation_type": "development validation",
}

(EVALUATION_DIR / "evaluation_settings.json").write_text(
    json.dumps(evaluation_record, indent=2)
)

print("Evaluation outputs:", EVALUATION_DIR)
print(json.dumps(EVALUATION_SETTINGS, indent=2))

Evaluation outputs: /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1/evaluation/selected_20261028_001307
{
  "batch_size": 16,
  "imgsz": 640,
  "conf_thres": 0.001,
  "iou_thres": 0.6,
  "max_det": 300,
  "task": "val",
  "device": "0",
  "workers": 2,
  "single_cls": false,
  "augment": false,
  "half": true
}


##Define Evalulation Function

In [11]:
import time

# Import the evaluator from the pinned YOLOv5 checkout.
sys.path.insert(0, str(YOLO_DIR))
import val as yolo_val

def evaluate_baseline(modality):
    """Evaluate one selected checkpoint using the shared settings."""
    assert modality in ["vis", "ir"]

    output_dir = EVALUATION_DIR / modality

    # Prediction text files can be appended to by the evaluator.
    # Refuse to reuse a directory so reruns cannot duplicate predictions.
    assert not output_dir.exists(), (
        f"Evaluation output already exists: {output_dir}. "
        "Do not rerun into the same directory."
    )

    # Point explicitly to the same frozen lists used during training.
    dataset_config = {
        "path": str(DATA_ROOT),
        "train": f"{modality}_train.txt",
        "val": f"{modality}_val.txt",
        "nc": 1,
        "names": ["person"],
    }

    print(f"Evaluating {modality.upper()}...", flush=True)
    start = time.perf_counter()

    results, per_class_ap, speeds = yolo_val.run(
        data=dataset_config,
        weights=str(CHECKPOINTS[modality]),
        project=str(EVALUATION_DIR),
        name=modality,
        exist_ok=False,
        save_txt=True,
        save_conf=True,
        save_hybrid=False,
        save_json=False,
        plots=True,
        verbose=True,
        **EVALUATION_SETTINGS,
    )

    # The first four returned values are P, R, AP50, and AP50–95.
    metric_values = np.asarray(results[:4], dtype=float)
    assert np.isfinite(metric_values).all(), "Non-finite evaluation metric."

    report = {
        "modality": modality,
        "checkpoint": str(CHECKPOINTS[modality]),
        "checkpoint_sha256": sha256_file(CHECKPOINTS[modality]),
        "precision": float(results[0]),
        "recall": float(results[1]),
        "AP50": float(results[2]),
        "AP50_95": float(results[3]),
        "per_class_AP50_95": np.asarray(per_class_ap).tolist(),
        "milliseconds_per_image": [float(value) for value in speeds],
        "elapsed_minutes": (time.perf_counter() - start) / 60,
        "validation_pairs": 1995,
        "settings": EVALUATION_SETTINGS,
    }

    (output_dir / "metrics.json").write_text(json.dumps(report, indent=2))

    print(f"\n{modality.upper()} EVALUATION COMPLETE")
    for metric in ["precision", "recall", "AP50", "AP50_95"]:
        print(f"{metric}: {100 * report[metric]:.2f}%")
    print("Saved predictions:", output_dir / "labels")
    print(f"Elapsed minutes: {report['elapsed_minutes']:.2f}")

    return report

##Evaluate VIS

In [12]:
# Load the selected VIS checkpoint and evaluate it without training
vis_evaluation = evaluate_baseline("vis")

Evaluating VIS...
YOLOv5 🚀 v7.0-556-gf9578796 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 68 layers, 7,012,822 parameters, 0 gradients, 15.8 GFLOPs


100%|██████████| 755k/755k [00:00<00:00, 25.2MB/s]


val: Scanning /content/eo_ir_recovery/verified_bundle/eo_ir_data/vis_val.cache... 1995 images, 0 backgrounds, 0 corrupt
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% ━━━━━━━━━━━━ 125/125 2.5s/it 5:18
                   all       1995       5246      0.915      0.755      0.853       0.44
Speed: 0.2ms pre-process, 4.0ms inference, 1.5ms NMS per image at shape (16, 3, 640, 640)
Results saved to /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1/evaluation/selected_20261028_001307/vis
1995 labels saved to /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1/evaluation/selected_20261028_001307/vis/labels

VIS EVALUATION COMPLETE
precision: 91.46%
recall: 75.51%
AP50: 85.28%
AP50_95: 44.01%
Saved predictions: /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1/evaluation/selected_20261028_001307/vis/labels
Elapsed minutes: 5.41


##Evaluate IR

In [17]:
from datetime import datetime, timezone

# Preserve any previous IR evaluation attempt instead of deleting it.
# Leave the completed VIS evaluation unchanged.
ir_output = EVALUATION_DIR / "ir"

if ir_output.exists():
    timestamp = datetime.now(timezone.utc).strftime(
        "%Y%m%d_%H%M%S_%f"
    )
    archived_output = EVALUATION_DIR / f"ir_previous_{timestamp}"

    ir_output.rename(archived_output)
    print("Previous IR attempt preserved at:", archived_output)

# The original IR output path is now available for a clean evaluation.
# This uses the same selected checkpoint and evaluation settings.
ir_evaluation = evaluate_baseline("ir")

Previous IR attempt preserved at: /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1/evaluation/selected_20261028_001307/ir_previous_20261028_004503_143387
Evaluating IR...
YOLOv5 🚀 v7.0-556-gf9578796 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 68 layers, 7,012,822 parameters, 0 gradients, 15.8 GFLOPs
val: Scanning /content/eo_ir_recovery/verified_bundle/eo_ir_data/ir_val.cache... 1995 images, 0 backgrounds, 0 corrupt
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% ━━━━━━━━━━━━ 125/125 1.6s/it 3:21
                   all       1995       5246      0.954      0.895       0.96      0.589
Speed: 0.2ms pre-process, 3.7ms inference, 1.4ms NMS per image at shape (16, 3, 640, 640)
Results saved to /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1/evaluation/selected_20261028_001307/ir
1995 labels saved to /content/drive/MyDrive/EO_IR_Cap

##Save Comparison and Check Consistency

In [18]:
# Read saved reports so this summary is traceable to persistent files.
reports = [
    json.loads((EVALUATION_DIR / modality / "metrics.json").read_text())
    for modality in ["vis", "ir"]
]

metrics = ["precision", "recall", "AP50", "AP50_95"]
comparison = pd.DataFrame(reports)[["modality"] + metrics]

# Keep decimal values in the saved CSV; convert only the displayed copy.
comparison.to_csv(EVALUATION_DIR / "comparison.csv", index=False)
percent_table = comparison.copy()
percent_table[metrics] *= 100

print("STANDALONE DEVELOPMENT-VALIDATION METRICS — PERCENT")
print(percent_table.to_string(index=False))

# Express IR-minus-VIS differences in percentage points.
indexed = comparison.set_index("modality")
differences = 100 * (
    indexed.loc["ir", metrics] - indexed.loc["vis", metrics]
)
print("\nIR MINUS VIS — PERCENTAGE POINTS")
print(differences.to_string())

# Compare against the previously selected training-validation rows.
training_columns = {
    "precision": "metrics/precision",
    "recall": "metrics/recall",
    "AP50": "metrics/mAP_0.5",
    "AP50_95": "metrics/mAP_0.5:0.95",
}

checks = []
for report in reports:
    modality = report["modality"]
    selected_row = training_selected_rows[modality]

    for metric, training_column in training_columns.items():
        checks.append({
            "modality": modality,
            "metric": metric,
            "training_record_percent": 100 * float(
                selected_row[training_column]
            ),
            "standalone_percent": 100 * report[metric],
            "difference_percentage_points": 100 * (
                report[metric] - float(selected_row[training_column])
            ),
        })

consistency = pd.DataFrame(checks)
consistency.to_csv(
    EVALUATION_DIR / "training_vs_standalone.csv",
    index=False,
)

print("\nTRAINING RECORD VERSUS STANDALONE EVALUATION")
print(consistency.to_string(index=False))
print("\nAll outputs:", EVALUATION_DIR)

STANDALONE DEVELOPMENT-VALIDATION METRICS — PERCENT
modality  precision    recall      AP50   AP50_95
     vis  91.457688 75.512388 85.283992 44.010308
      ir  95.377063 89.534884 96.016004 58.850929

IR MINUS VIS — PERCENTAGE POINTS
precision     3.919376
recall       14.022496
AP50         10.732012
AP50_95      14.840621

TRAINING RECORD VERSUS STANDALONE EVALUATION
modality    metric  training_record_percent  standalone_percent  difference_percentage_points
     vis precision                   91.513           91.457688                     -0.055312
     vis    recall                   75.432           75.512388                      0.080388
     vis      AP50                   85.299           85.283992                     -0.015008
     vis   AP50_95                   44.023           44.010308                     -0.012692
      ir precision                   95.437           95.377063                     -0.059937
      ir    recall                   89.554           89.53488

##Inspect Learning Hisories

In [19]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

# Save diagnostic figures beside the verified evaluation results.
history_dir = EVALUATION_DIR / "learning_history"
history_dir.mkdir(parents=True, exist_ok=True)

histories = {}
combined = []

for modality in ["vis", "ir"]:
    source = EXPERIMENT_DIR / f"{modality}_baseline/results.csv"
    history = pd.read_csv(source)
    history.columns = history.columns.str.strip()

    # Display epochs as 1–10 while preserving the original CSV numbering.
    assert list(history["epoch"].astype(int)) == list(range(10))
    history["display_epoch"] = history["epoch"].astype(int) + 1
    histories[modality] = history

    # Preserve both original histories in one clearly labeled CSV.
    labeled = history.copy()
    labeled.insert(0, "modality", modality)
    combined.append(labeled)

pd.concat(combined, ignore_index=True).to_csv(
    history_dir / "combined_learning_history.csv",
    index=False,
)

# Loss curves diagnose learning behavior; validation AP curves measure
# detection performance. Do not interpret training loss as detection accuracy.
panels = [
    ("train/box_loss", "Training box loss", "Loss", False),
    ("train/obj_loss", "Training objectness loss", "Loss", False),
    ("metrics/mAP_0.5", "Development-validation AP50", "AP (%)", True),
    ("metrics/mAP_0.5:0.95", "Development-validation AP50–95", "AP (%)", True),
]

styles = {
    "vis": {"color": "#0072B2", "linestyle": "-", "marker": "o"},
    "ir": {"color": "#D55E00", "linestyle": "--", "marker": "s"},
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for axis, (column, title, ylabel, percentage) in zip(axes.flat, panels):
    for modality, history in histories.items():
        values = history[column] * (100 if percentage else 1)

        axis.plot(
            history["display_epoch"],
            values,
            label=modality.upper(),
            **styles[modality],
        )

    axis.set_title(title)
    axis.set_xlabel("Completed training epoch")
    axis.set_ylabel(ylabel)
    axis.set_xticks(range(1, 11))
    axis.grid(alpha=0.25)
    axis.legend()

    # Use a shared percentage scale for both validation panels.
    if percentage:
        axis.set_ylim(0, 100)

fig.suptitle("VIS and IR baseline learning histories")
fig.tight_layout(rect=[0, 0, 1, 0.96])

figure_path = history_dir / "baseline_learning_histories.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

# Summarize the selected peak and final epoch without changing selection.
print("AP50–95 HISTORY SUMMARY")
for modality, history in histories.items():
    ranked = history.sort_values(
        ["metrics/mAP_0.5:0.95", "epoch"],
        ascending=[False, True],
    )
    best = ranked.iloc[0]
    final = history.iloc[-1]

    print(
        f"{modality.upper()}: "
        f"epoch 1 = {100 * history.iloc[0]['metrics/mAP_0.5:0.95']:.2f}%; "
        f"best = {100 * best['metrics/mAP_0.5:0.95']:.2f}% "
        f"at epoch {int(best['display_epoch'])}; "
        f"epoch 10 = {100 * final['metrics/mAP_0.5:0.95']:.2f}%"
    )

print("\nFigure saved:", figure_path)

AP50–95 HISTORY SUMMARY
VIS: epoch 1 = 36.91%; best = 44.02% at epoch 9; epoch 10 = 43.17%
IR: epoch 1 = 50.70%; best = 58.84% at epoch 10; epoch 10 = 58.84%

Figure saved: /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1/evaluation/selected_20261028_001307/learning_history/baseline_learning_histories.png


#**Paired Ground-Truth and Prediction Examples**
##**Purpose**
Visually compare the selected VIS and IR detectors on matching
development-validation image pairs.

##**Sampling**
Select six pairs reproducibly using seed 42, covering the available
validation filename-prefix groups. These are illustrative examples,
not a separate performance estimate.

##**Figure layout**
Top row: VIS and IR images with ground-truth boxes.
Bottom row: the same images with detector predictions.

##**Display settings**
Show predictions with confidence at least 0.25.
This display threshold is separate from the evaluator's confidence
sweep and selected precision/recall operating point.

##**Review**
Identify missed pedestrians, false detections, and box-placement errors.
Do not infer correctness from the number of predicted boxes alone.

In [20]:
from pathlib import Path
import json
import random

# Use the active evaluation folder and locally extracted dataset.
# If the runtime reset, rerun the startup cells before continuing.
data_root = Path(DATA_ROOT)
evaluation_dir = Path(EVALUATION_DIR)

assert data_root.is_dir(), "Rerun the data-loading startup cells."
assert evaluation_dir.is_dir(), "The evaluation folder is unavailable."

# Confirm that both completed evaluation reports and prediction folders exist.
for modality in ["vis", "ir"]:
    assert (evaluation_dir / modality / "metrics.json").is_file(), (
        f"{modality.upper()} evaluation is not recorded as complete."
    )
    assert (evaluation_dir / modality / "labels").is_dir(), (
        f"{modality.upper()} saved predictions are missing."
    )

# Confirm that both modalities use the same validation membership.
validation_ids = {}
for modality in ["vis", "ir"]:
    entries = (data_root / f"{modality}_val.txt").read_text().split()
    validation_ids[modality] = {
        Path(entry).stem for entry in entries
    }

assert validation_ids["vis"] == validation_ids["ir"]
assert len(validation_ids["vis"]) == 1995

FIGURE_DIR = evaluation_dir / "paired_examples"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
sample_file = FIGURE_DIR / "sample_ids.json"

if sample_file.exists():
    # Reuse the recorded sample when rerunning this cell.
    # This prevents repeatedly resampling until examples look favorable.
    sample_record = json.loads(sample_file.read_text())
    selected_ids = sample_record["pair_ids"]
else:
    rng = random.Random(42)

    # Group by filename prefix, without claiming prefixes guarantee
    # independent physical scenes.
    groups = {}
    for pair_id in sorted(validation_ids["vis"]):
        groups.setdefault(pair_id[:2], []).append(pair_id)

    prefix_order = sorted(groups)
    rng.shuffle(prefix_order)

    for members in groups.values():
        rng.shuffle(members)

    # Select in rounds so the sample covers the available prefix groups.
    selected_ids = []
    while len(selected_ids) < 6:
        for prefix in prefix_order:
            if groups[prefix]:
                selected_ids.append(groups[prefix].pop())
            if len(selected_ids) == 6:
                break

    sample_record = {
        "seed": 42,
        "method": "Random selection in rounds across validation ID prefixes",
        "pair_ids": selected_ids,
        "display_confidence": 0.25,
        "purpose": "Illustrative sample; not a performance estimate",
    }
    sample_file.write_text(json.dumps(sample_record, indent=2))

assert len(selected_ids) == 6
assert len(set(selected_ids)) == 6
assert set(selected_ids) <= validation_ids["vis"]

DISPLAY_CONFIDENCE = 0.25

print("Selected validation pairs:", selected_ids)
print("Display confidence:", DISPLAY_CONFIDENCE)
print("Sample record:", sample_file)
print("Figures will be saved to:", FIGURE_DIR)

Selected validation pairs: ['150558', '140043', '180082', '050126', '150231', '140316']
Display confidence: 0.25
Sample record: /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1/evaluation/selected_20261028_001307/paired_examples/sample_ids.json
Figures will be saved to: /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1/evaluation/selected_20261028_001307/paired_examples


##Define Figure Functions

In [21]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

def read_yolo_boxes(path, predictions=False):
    """Read normalized YOLO boxes, optionally including confidence."""
    path = Path(path)

    if not path.is_file():
        # The evaluator may omit a prediction file when there are no detections.
        # Missing ground-truth files, however, should stop the review.
        if predictions:
            return []
        raise FileNotFoundError(f"Ground-truth label missing: {path}")

    boxes = []

    for line in path.read_text().splitlines():
        if not line.strip():
            continue

        values = [float(value) for value in line.split()]
        expected_columns = 6 if predictions else 5
        assert len(values) == expected_columns, (path, line)
        assert np.isfinite(values).all(), (path, line)

        class_id, cx, cy, width, height = values[:5]
        assert class_id == 0, f"Unexpected class in {path}"

        confidence = values[5] if predictions else None

        # Filter only the displayed predictions, not the saved evaluator output.
        if predictions and confidence < DISPLAY_CONFIDENCE:
            continue

        boxes.append((cx, cy, width, height, confidence))

    return boxes


def draw_panel(axis, image_path, boxes, title, predictions=False):
    """Draw one original image with either annotations or predictions."""
    with Image.open(image_path) as image:
        image = image.convert("RGB")
        width, height = image.size
        axis.imshow(image)

    # Use different line styles as well as colors to distinguish box types.
    color = "#D55E00" if predictions else "#009E73"
    linestyle = "--" if predictions else "-"

    for cx, cy, box_width, box_height, confidence in boxes:
        left = (cx - box_width / 2) * width
        top = (cy - box_height / 2) * height

        axis.add_patch(
            Rectangle(
                (left, top),
                box_width * width,
                box_height * height,
                fill=False,
                edgecolor=color,
                linewidth=1.8,
                linestyle=linestyle,
            )
        )

        if predictions:
            # Confidence describes the prediction, not its verified correctness.
            axis.text(
                left,
                max(0, top - 3),
                f"{confidence:.2f}",
                fontsize=9,
                color="white",
                bbox={
                    "facecolor": color,
                    "edgecolor": "none",
                    "alpha": 0.85,
                    "pad": 1,
                },
            )

    axis.set_title(title)
    axis.axis("off")


def make_paired_figure(pair_id):
    """Save a four-panel comparison for one validation pair."""
    ground_truth = {}
    predictions = {}

    for modality in ["vis", "ir"]:
        ground_truth[modality] = read_yolo_boxes(
            data_root / modality / "labels" / f"{pair_id}.txt"
        )
        predictions[modality] = read_yolo_boxes(
            evaluation_dir / modality / "labels" / f"{pair_id}.txt",
            predictions=True,
        )

    # Both registered modalities should use matching ground-truth labels.
    assert ground_truth["vis"] == ground_truth["ir"], (
        f"Ground-truth labels differ for pair {pair_id}"
    )

    fig, axes = plt.subplots(2, 2, figsize=(15, 11))

    for column, modality in enumerate(["vis", "ir"]):
        image_path = (
            data_root / modality / "images" / f"{pair_id}.jpg"
        )
        assert image_path.is_file(), image_path

        draw_panel(
            axes[0, column],
            image_path,
            ground_truth[modality],
            f"{modality.upper()} ground truth "
            f"— {len(ground_truth[modality])} annotated pedestrians",
        )
        draw_panel(
            axes[1, column],
            image_path,
            predictions[modality],
            f"{modality.upper()} predictions "
            f"— {len(predictions[modality])} boxes at confidence ≥ "
            f"{DISPLAY_CONFIDENCE:.2f}",
            predictions=True,
        )

    fig.suptitle(
        f"Validation pair {pair_id} — ground truth above, predictions below"
    )
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    output_path = FIGURE_DIR / f"{pair_id}_comparison.png"
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    return {
        "pair_id": pair_id,
        "prefix": pair_id[:2],
        "ground_truth_pedestrians": len(ground_truth["vis"]),
        "vis_displayed_predictions": len(predictions["vis"]),
        "ir_displayed_predictions": len(predictions["ir"]),
        "display_confidence": DISPLAY_CONFIDENCE,
        "figure_path": str(output_path),
    }

##Generate 6 Figures

In [22]:
import pandas as pd

# Generate figures from saved predictions; no model execution occurs here.
example_records = []

for pair_id in selected_ids:
    print(f"Generating comparison for {pair_id}...", flush=True)
    example_records.append(make_paired_figure(pair_id))

# Save an index so each figure is traceable to its pair and display settings.
example_index = pd.DataFrame(example_records)
index_path = FIGURE_DIR / "example_index.csv"
example_index.to_csv(index_path, index=False)

print("\nPAIRED EXAMPLES COMPLETE")
print(example_index.to_string(index=False))
print("\nFigure folder:", FIGURE_DIR)
print("Index:", index_path)

Generating comparison for 150558...
Generating comparison for 140043...
Generating comparison for 180082...
Generating comparison for 050126...
Generating comparison for 150231...
Generating comparison for 140316...

PAIRED EXAMPLES COMPLETE
pair_id prefix  ground_truth_pedestrians  vis_displayed_predictions  ir_displayed_predictions  display_confidence                                                                                                                                               figure_path
 150558     15                         3                          3                         4                0.25 /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1/evaluation/selected_20261028_001307/paired_examples/150558_comparison.png
 140043     14                         4                          3                         3                0.25 /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1/evaluation/selected_20261028

#**Interpretation of paired examples**

Examples were selected reproducibly using seed 42 across validation
filename-prefix groups. Predictions are displayed at confidence ≥ 0.25.

These examples illustrate detection behavior rather than estimate
dataset-wide performance. They show successful detections, shared misses,
and potential duplicate predictions in both modalities.

The standalone validation metrics establish IR's aggregate advantage.
The examples show that this advantage does not imply superiority on
every individual image.

No fusion method has been evaluated yet. Our equal-weight fusion baseline
remains a later experiment.